# Haar Cascade Notes

## Overview
- Haar Cascade is a **machine learning-based object detection algorithm**.
- Introduced by **Paul Viola** and **Michael Jones** in 2001.
- Commonly used for **face, eye, smile, person, and car detection**.
- Suitable for systems with **limited computational resources**.

## Core Concepts
### 1. Haar-like Features
- Rectangular patterns used to compare pixel intensities.
- Detect:
  - Edges
  - Lines
  - Textures
- Common feature types:
  - Edge features
  - Line features
  - Four-rectangle features

### 2. Cascade of Classifiers
- Multiple weak classifiers arranged sequentially.
- If one stage fails, remaining stages are skipped.
- Improves computational efficiency.
- Common boosting algorithms:
  - Discrete AdaBoost (DAB)
  - Real AdaBoost (RAB)
  - LogitBoost (LB)
  - Gentle AdaBoost (GAB)

## How Haar Features Work
1. Divide image into rectangular regions.
2. Sum pixel intensities in white and black regions.
3. Compute the difference.
4. Compare the result with trained Haar features.
5. If similarity is high, feature is considered detected.

## Detection Pipeline
1. Convert image to grayscale.
2. Slide Haar features across image regions.
3. Apply cascade stages.
4. If all stages pass:
   - Object detected.
   - Return bounding box `(x, y, w, h)`.

## Applications
- Face detection in camera apps.
- Smile detection.
- Eye detection.
- Security surveillance.
- Person detection in robotics.
- Car detection systems.

## Important Parameters
### scaleFactor
- Controls image scaling during detection.
- Example: `1.3`
- Helps detect objects at different sizes.

### minNeighbors
- Controls detection strictness.
- Higher value:
  - Fewer false positives.
  - More strict detection.
- Example:
  - `5` = balanced
  - `20` = stricter detection

## OpenCV Haar Cascade Files
Examples:
- `haarcascade_frontalface_default.xml`
- `haarcascade_eye.xml`
- `haarcascade_smile.xml`

Location:
```python
cv2.data.haarcascades
```

## Workflow Used in This Notebook
### Face Detection
1. Capture webcam frames.
2. Convert frame to grayscale.
3. Run:
```python
face_cascade.detectMultiScale()
```
4. Draw bounding boxes.
5. Display detection label.

### Face + Eye + Mouth Detection
1. Detect face first.
2. Extract ROI (Region of Interest).
3. Detect eyes inside face ROI.
4. Detect mouth inside face ROI.
5. Draw separate rectangles and labels.

## Advantages
- Fast.
- Lightweight.
- Works in real time.
- No GPU required.

## Limitations
- Less accurate than deep learning models.
- Sensitive to lighting conditions.
- Can generate false positives.
- Works best with frontal faces.


In [1]:
import cv2

In [2]:
# Face Detection

cam = cv2.VideoCapture(0)

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

while True:
    ret, frame = cam.read()

    if not ret:
        print("Failed to grab frame")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

    for (x,y,w,h) in faces:

        cv2.rectangle(frame, (x,y), (x+w, y+h), (255, 0, 0), 2)

        cv2.putText(frame, "Face Detected", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2, cv2.LINE_AA)

    cv2.imshow('Camera', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()

In [5]:
# face + eye + mouth

cam = cv2.VideoCapture(0)

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
mouth_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_smile.xml')

while True:
    ret, frame = cam.read()

    if not ret:
        print("Failed to grab frame")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

    for (x,y,w,h) in faces:

        cv2.rectangle(frame, (x,y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(frame, "Face Detected", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2, cv2.LINE_AA)

        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]

        eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.3, minNeighbors=5)
        mouths = mouth_cascade.detectMultiScale(roi_gray, scaleFactor=1.8, minNeighbors=20)

        for (ex,ey, ew, eh) in eyes:
            cv2.rectangle(frame, (x+ex,y+ey), (x+ex+ew, y+ey+eh), (255, 0, 0), 2)
            cv2.putText(frame, "Eye", (x+ex,(y+ey) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2, cv2.LINE_AA)
        
        for (mx,my, mw, mh) in mouths:
            cv2.rectangle(frame, (x+mx,y+my), (x+mx+mw, y+my+mh), (255, 0, 0), 2)
            cv2.putText(frame, "Mouth", (x+mx,(y+my) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2, cv2.LINE_AA)
        

    cv2.imshow('Camera', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()